# Customer Churn Prediction (Codveda)

**Project goal:** predict customer churn early enough to support retention action.

I structured this notebook as a portfolio-ready case study: problem framing, EDA, modeling, evaluation, and deployment handoff.

## Business Context

Telecom churn is costly because acquiring new customers is expensive. In this project, I use a model to flag likely churners early so retention teams can act before cancellation.

## 1) Setup and Imports

In [ ]:
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    classification_report,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.titleweight'] = 'bold'

ROOT = Path('.').resolve()
TRAIN_PATH = ROOT / 'churn-bigml-80.csv'
TEST_PATH = ROOT / 'churn-bigml-20.csv'
MODEL_PATH = ROOT / 'models' / 'churn_model.joblib'
MODEL_PATH.parent.mkdir(exist_ok=True)


## 2) Load Data and Snapshot

In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print(f'Train shape: {train_df.shape}')
print(f'Test shape: {test_df.shape}')

display(train_df.head())


## 3) Data Quality Review

Checks below confirm schema health before modeling: data types, missing values, and target balance.

In [ ]:
summary = pd.DataFrame({
    'dtype': train_df.dtypes.astype(str),
    'missing': train_df.isna().sum(),
    'missing_pct': (train_df.isna().mean() * 100).round(2),
})
display(summary)


## 4) Target Distribution (Class Imbalance Check)

In [ ]:
target_counts = train_df['Churn'].value_counts()
target_pct = (train_df['Churn'].value_counts(normalize=True) * 100).round(2)

display(pd.DataFrame({'count': target_counts, 'percent': target_pct}))

ax = sns.countplot(data=train_df, x='Churn', hue='Churn', palette=['#4C78A8', '#E45756'], legend=False)
ax.set_title('Churn Distribution - Train Set')
ax.set_xlabel('Churn')
ax.set_ylabel('Customers')
plt.tight_layout()
plt.show()


## 5) Behavioral Signals in EDA

These quick views highlight practical churn drivers often seen in telecom data.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(
    data=train_df, x='International plan', y='Churn', estimator=np.mean, errorbar=None,
    palette=['#72B7B2', '#F58518'], ax=axes[0]
)
axes[0].set_title('Churn Rate by International Plan')
axes[0].set_ylabel('Churn Rate')
axes[0].set_xlabel('International plan')

sns.barplot(
    data=train_df, x='Customer service calls', y='Churn', estimator=np.mean, errorbar=None,
    color='#54A24B', ax=axes[1]
)
axes[1].set_title('Churn Rate vs Customer Service Calls')
axes[1].set_ylabel('Churn Rate')
axes[1].set_xlabel('Customer service calls')

plt.tight_layout()
plt.show()


In [ ]:
numeric_cols_for_corr = train_df.select_dtypes(include='number').columns

plt.figure(figsize=(12, 8))
corr = train_df[numeric_cols_for_corr].corr()
sns.heatmap(corr, cmap='RdBu_r', center=0, linewidths=0.2)
plt.title('Correlation Heatmap (Numeric Features)')
plt.tight_layout()
plt.show()


## 6) Feature Engineering + Model Training

Approach:
- One-hot encode categorical variables (`State`, plan flags).
- Pass numeric features directly.
- Train RandomForest with class balancing for minority churn class.

In [ ]:
TARGET_COL = 'Churn'
CATEGORICAL_COLS = ['State', 'International plan', 'Voice mail plan']
FEATURE_COLS = [c for c in train_df.columns if c != TARGET_COL]
NUMERIC_COLS = [c for c in FEATURE_COLS if c not in CATEGORICAL_COLS]

X_train = train_df[FEATURE_COLS]
y_train = train_df[TARGET_COL].astype(bool)
X_test = test_df[FEATURE_COLS]
y_test = test_df[TARGET_COL].astype(bool)

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL_COLS),
    ('num', 'passthrough', NUMERIC_COLS),
])

model = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=42,
    n_jobs=1,
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', model),
])

pipeline.fit(X_train, y_train)
print('Model training complete.')


## 7) Evaluation on Holdout Test Set

In [ ]:
preds = pipeline.predict(X_test)
probs = pipeline.predict_proba(X_test)[:, 1]

print('Classification Report:')
print(classification_report(y_test, preds, digits=4))
print(f'ROC AUC: {roc_auc_score(y_test, probs):.4f}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ConfusionMatrixDisplay.from_predictions(y_test, preds, cmap='Blues', ax=axes[0])
axes[0].set_title('Confusion Matrix')

RocCurveDisplay.from_predictions(y_test, probs, ax=axes[1])
axes[1].set_title('ROC Curve')

plt.tight_layout()
plt.show()


## 8) Save Model for API Integration

Saved artifact contains:
- `pipeline`: preprocessing + model
- `feature_columns`: expected request/order schema for API inference

In [ ]:
bundle = {'pipeline': pipeline, 'feature_columns': FEATURE_COLS}
joblib.dump(bundle, MODEL_PATH)
print(f'Model saved to: {MODEL_PATH.name}')


## 9) Single-Record Inference Example

In [ ]:
sample = X_test.iloc[[0]].copy()
sample_prob = float(pipeline.predict_proba(sample)[0, 1])
sample_pred = bool(pipeline.predict(sample)[0])

display(sample)
print(f'Predicted churn: {sample_pred}')
print(f'Predicted churn probability: {sample_prob:.4f}')


## 10) Portfolio Talking Points

1. Delivered an end-to-end supervised ML workflow from raw data to API-ready model artifact.
2. Addressed class imbalance with weighted training and evaluated with ROC AUC beyond accuracy.
3. Designed reusable preprocessing pipeline to reduce train-serving skew risk.
4. Prepared deployment handoff via `models/churn_model.joblib` and FastAPI endpoint compatibility.

## 11) My Next Iteration Plan

1. Add threshold tuning based on business cost (false churn alert vs missed churn).
2. Compare with Logistic Regression and Gradient Boosting for interpretability/performance tradeoff.
3. Add SHAP or feature importance reporting for stakeholder explainability.